# AI-RVC 一键 AI 翻唱 (Colab)

[Open in Colab](https://colab.research.google.com/github/mason369/AI-RVC/blob/master/AI_RVC_Colab.ipynb)

这是 AI-RVC 的 Google Colab 运行入口。Notebook 会创建独立 Python 3.10 环境，再运行项目安装脚本，避免 Colab 默认 Python 版本变化影响 `fairseq==0.12.2`。

当前处理链路模型：

| 环节 | 默认模型 |
|------|----------|
| 人声 / 伴奏分离 | `hybrid:leap_xe90_vocals+polarformer62_instrumental`：同一首原始整曲分别交给 Leap XE 90 提取人声、BS PolarFormer public ONNX 62 提取纯伴奏 |
| 卡拉OK分离 | [MVSep 9205](https://www.mvsep.com/quality_checker/entry/9205) / `ensemble:mvsep_9205_avg`：三模型 `avg_wave` 处理原始整曲，输出主唱与带和声伴奏；Leap 人声减去主唱得到纯和声，PolarFormer 纯伴奏单独保留 |
| 去混响 / DeEcho | [dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt](https://pypi.org/project/audio-separator/) |
| 内容特征 | [hubert_base.pt](https://huggingface.co/lj1995/VoiceConversionWebUI/blob/main/hubert_base.pt) |
| F0 音高 | [rmvpe.pt](https://huggingface.co/lj1995/VoiceConversionWebUI/blob/main/rmvpe.pt) |
| VC 推理 | [RVC v2](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI) `.pth` + 可选 [FAISS](https://github.com/facebookresearch/faiss) `.index` |

默认输出：`backing_vocals.wav` 是纯和声，`accompaniment.wav` 是 MVSep `Back+Instrumental`（带和声伴奏），`accompaniment_without_harmony.wav` 是 PolarFormer 纯伴奏。

使用前请在 Colab 菜单选择：`代码执行程序 -> 更改运行时类型 -> T4 GPU`，然后按顺序运行每个单元格。

## 1. 检查 Colab 运行时

这一步会显示当前 GPU 和 Colab 默认 Python。后面安装时会使用独立 Python 3.10 环境。

In [ ]:
import sys
import subprocess

print('Colab 默认 Python:', sys.version)
print('\nGPU 信息:')
subprocess.run(['nvidia-smi'], check=True)

## 2. 克隆或刷新仓库

如果 `/content/AI-RVC` 已存在，会重置到 GitHub `master` 分支，确保 Colab 环境和仓库最新代码一致。

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d AI-RVC/.git ]; then
  git -C AI-RVC fetch origin master
  git -C AI-RVC reset --hard origin/master
else
  git clone https://github.com/mason369/AI-RVC.git AI-RVC
fi
cd /content/AI-RVC
git rev-parse --short HEAD
ls -la

## 3. 创建 Python 3.10 环境并安装依赖

这个步骤会安装 `uv`，用它准备 Python 3.10，然后调用项目自带 `install.py --no-run`。如果任何依赖安装失败，单元格会直接停止并显示错误。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC

sudo apt-get update -y
sudo apt-get install -y ffmpeg build-essential python3-dev curl git

if ! command -v uv >/dev/null 2>&1; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"
uv --version
uv python install 3.10
uv venv --seed --python 3.10 venv310
PY=/content/AI-RVC/venv310/bin/python
$PY --version
$PY -m pip --version
$PY -m pip install --upgrade pip
$PY -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126
$PY - <<'PY'
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU PyTorch install completed but CUDA is not available.')
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
PY
uv run --python 3.10 python install.py --no-run

## 4. 检查依赖与默认模型配置

这里会检查关键包、Python 版本、CUDA、Colab 默认分离模型、卡拉OK模型和 DeEcho 模型是否符合当前项目配置。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY - <<'PY'
import importlib.metadata as md
import sys
import torch
from infer.separator import (
    ROFORMER_DEFAULT_MODEL,
    KARAOKE_DEFAULT_MODEL,
    ROFORMER_DEREVERB_DEFAULT_MODEL,
)

assert sys.version_info[:2] == (3, 10), sys.version
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Colab runtime did not provide CUDA. Please switch runtime type to T4 GPU and rerun.')

expected = {
    'audio-separator': '0.44.1',
    'onnxruntime-gpu': '1.18.0',
    'einops': '0.7.0',
    'PyYAML': '6.0',
    'fairseq': '0.12.2',
    'gradio': '5.49.1',
}
for dist, minimum in expected.items():
    version = md.version(dist)
    print(f'{dist}: {version}')

assert ROFORMER_DEFAULT_MODEL == 'hybrid:leap_xe90_vocals+polarformer62_instrumental', ROFORMER_DEFAULT_MODEL
assert KARAOKE_DEFAULT_MODEL == 'ensemble:mvsep_9205_avg', KARAOKE_DEFAULT_MODEL
assert ROFORMER_DEREVERB_DEFAULT_MODEL == 'dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt', ROFORMER_DEREVERB_DEFAULT_MODEL

import faiss  # noqa: F401
import onnxruntime
if 'CUDAExecutionProvider' not in onnxruntime.get_available_providers():
    raise RuntimeError('Colab ONNX Runtime did not provide CUDAExecutionProvider.')
import einops  # noqa: F401
import yaml  # noqa: F401
from audio_separator.separator import Separator  # noqa: F401
import fairseq  # noqa: F401
print('Dependency and model default checks passed.')
PY

## 5. 下载基础模型和默认分离模型

下载 [HuBERT](https://arxiv.org/abs/2106.07447)、[RMVPE](https://arxiv.org/abs/2306.15412)、[UVR5 HP2](https://huggingface.co/lj1995/VoiceConversionWebUI/blob/main/uvr5_weights/HP2_all_vocals.pth)、Leap XE vocals、BS PolarFormer pure accompaniment、MVSep 9205 子模型和 RoFormer De-Reverb，并准备内置官方 RVC 源码。模型会保存到仓库的 `assets/` 目录，官方源码会放在 `_official_rvc/`。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY tools/download_models.py
$PY - <<'PY'
from tools.download_models import REQUIRED_MODELS, check_model, check_required_default_separator_models, get_missing_default_separator_model_files
missing = [name for name in REQUIRED_MODELS if not check_model(name)]
if missing:
    raise RuntimeError('Missing required models: ' + ', '.join(missing))
if not check_required_default_separator_models():
    raise RuntimeError('Missing default separator models: ' + ', '.join(get_missing_default_separator_model_files()))
print('Required models are present:', ', '.join(REQUIRED_MODELS))
print('Default separator models are present.')
PY

## 6. 可选：查看或下载声线模型

处理链路模型已经在前面准备好。这里的 `.pth` 声线模型是 RVC 目标音色，不属于分离 / F0 / HuBERT 等处理模型。可以先下载一个示例模型，也可以跳过后在 Web UI 中下载。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY - <<'PY'
from tools.character_models import list_available_characters, list_available_series
print('Available series:')
for series in list_available_series():
    print(' -', series)
chars = list_available_characters()
print('\nFirst 20 voice models:')
for char in chars[:20]:
    print(f" - {char['name']}: {char.get('display', char.get('description', ''))}")
print('\nTotal:', len(chars))
PY

In [ ]:
# Optional: download one example voice model.
# Change character_name as needed, then run this cell.

# %%bash
# set -euo pipefail
# cd /content/AI-RVC
# PY=/content/AI-RVC/venv310/bin/python
# $PY - <<'PY'
# from tools.character_models import download_character_model
# character_name = 'rin'
# if not download_character_model(character_name):
#     raise RuntimeError(f'Failed to download voice model: {character_name}')
# print(f'Downloaded voice model: {character_name}')
# PY

## 7. 可选：试跑完整处理流程

上传一段含有人声的短音乐文件到 `/content/input_audio.*`，或把自己的文件路径填到 `INPUT_AUDIO`。这个步骤会下载示例声线模型 `rin`，并依次试跑默认 hybrid 分离链路（非 WAV 统一解码后，同一份 PCM 分别交给 Leap XE vocals 与 BS PolarFormer pure accompaniment）、[MVSep 9205 Karaoke](https://www.mvsep.com/quality_checker/entry/9205)（三模型 `avg_wave` 处理原始整曲，输出主唱与带和声伴奏，并从 Leap 人声差分得到纯和声）、[Demucs](https://github.com/facebookresearch/demucs)、[UVR5](https://github.com/Anjok07/ultimatevocalremovergui)、官方 1:1 等处理模式。若某个模式失败，Notebook 会停在对应步骤并显示错误信息。


In [ ]:
# Upload a short vocal music file, then set INPUT_AUDIO to its path.
# Example after upload: INPUT_AUDIO = '/content/input_audio.wav'
INPUT_AUDIO = '/content/input_audio.wav'

from pathlib import Path
if not Path(INPUT_AUDIO).exists():
    raise FileNotFoundError(
        f'Missing test audio: {INPUT_AUDIO}. Upload a short vocal music file to this path before running the matrix.'
    )
print('Matrix input:', INPUT_AUDIO, Path(INPUT_AUDIO).stat().st_size, 'bytes')


In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY - <<'PY'
from tools.character_models import download_character_model
if not download_character_model('rin'):
    raise RuntimeError('Failed to download voice model: rin')
print('Voice model ready: rin')
PY


In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY tools/run_mode_matrix.py \
  --input /content/input_audio.wav \
  --output-dir /content/AI-RVC/outputs/mode_matrix_colab \
  --require-cuda


## 8. 启动 Web UI

运行后 Colab 会输出 Gradio 公共链接。打开链接即可使用 AI-RVC Web UI。此单元格会再次检查环境和必需模型。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY run.py --host 0.0.0.0 --port 7860 --share

## 常见问题

| 问题 | 处理方式 |
|------|----------|
| `nvidia-smi` 失败 | 在 Colab 菜单选择 GPU 运行时后重新运行 |
| `fairseq` 安装失败 | 确认本 notebook 的第 3 步使用了 Python 3.10 环境 |
| `CUDA out of memory` | 换短音频，或在 UI 中降低处理负载 / 切换分离器 |
| 模型下载失败 | 重新运行第 5 步；如果 Hugging Face 限流，可稍后再试或配置 token |

项目代码按 MIT License 发布。模型、音频、角色形象和第三方组件仍受各自许可与权利约束；使用者须确认拥有输入内容和生成结果所需的授权，不得用于欺诈、冒充或侵权。